In [0]:
# Read JSON array format (multiLine JSON)
df = spark.read.option("multiLine", "true").json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/")
df.display()

## The desired pipeline structure we are attempting to create is here
![image_1775431627249.png](./image_1775431627249.png "image_1775431627249.png")

In [0]:
spark.sql(f'''
          SELECT * 
          FROM JSON.`/Volumes/workspace/default/test_volume/test_directory/orders/`
          ''').display()

### Set this batch job to be run every day
(this is not incremental job as the table is not streaming table but static - meaning that each batch will update and process the entire table)

In [0]:
%sql
-- JSON -> Bronze
CREATE OR REPLACE TABLE `data-pipeline`.default.orders_bronze
AS 
SELECT *,
  current_timestamp() AS processing_tinme,
  _metadata.file_name AS source_file
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/orders/',
  format => 'json'
);
    
-- Bronze -> Silver
-- read the entire bronze table each time
CREATE OR REPLACE TABLE `data-pipeline`.default.orders_silver
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp, 
  customer_id,
  notifications
FROM `data-pipeline`.default.orders_bronze;   

-- Silver -> Gold
-- Aggregate the silver each time the query is executed.
CREATE OR REPLACE VIEW `data-pipeline`.default.orders_by_date_vw     
AS 
SELECT 
  date(order_timestamp) AS order_date, 
  count(*) AS total_daily_orders
FROM `data-pipeline`.default.orders_silver                               
GROUP BY date(order_timestamp);

In [0]:
%sql
Select * from `data-pipeline`.default.orders_bronze limit 5

In [0]:
%sql
Select * from `data-pipeline`.default.orders_silver limit 5

In [0]:
%sql
Select * from `data-pipeline`.default.orders_by_date_vw limit 5

### Set the previous batch pipeline as a Lakeflow Declarative Pipelines

In [0]:
%sql
SELECT * 
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/orders/01.json',
  format => 'json'
)

In [0]:
%sql
SELECT * FROM `data-pipeline`.bronze_db.orders_bronze

In [0]:
spark.sql(f'list "/Volumes/workspace/default/test_volume/test_directory/orders/" ').display()

## Deploy a Pipeline to Production

![image_1775789583381.png](./image_1775789583381.png "image_1775789583381.png")

In [0]:
%sql
-- join the order and status table
with orders as (
  select *
  from read_files('/Volumes/workspace/default/test_volume/test_directory/orders/',
  format => 'json')
),
status as (
  select *
  from read_files('/Volumes/workspace/default/test_volume/test_directory/status/',
  format => 'json')
) -- join the views to get the order history with status
select 
  orders.order_id,
  timestamp(orders.order_timestamp) as order_timestamp,
  status.order_status,
  timestamp(status.status_timestamp) as order_status_timestamp
from orders
  inner join status
  on orders.order_id = status.order_id
order by order_id, order_status_timestamp

copy 5 orders and status files from volume to the current default schema

In [0]:
# copy the stream json files from the shared delta lake to the default workspace volume
def copy_files(copy_from_path, copy_to_path, n):
    # List files in source directory
    files = dbutils.fs.ls(copy_from_path)

    # Copy first n files
    for i, file in enumerate(files[:n]):
        if file.isFile():
            dbutils.fs.cp(file.path, copy_to_path + file.name)
            print(f"Copied {file.name}")

    print(f"\nCopied {min(n, len(files))} files")

In [0]:
copy_files(copy_from_path = "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json", 
           copy_to_path = "/Volumes/workspace/default/test_volume/test_directory/orders/", 
           n = 5)

copy_files(copy_from_path = "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/stream_json", 
           copy_to_path = "/Volumes/workspace/default/test_volume/test_directory/status/", 
           n = 5)

Check the pipeline event log (provided that you turned this functionality on in the pipeline advenced configuration)

In [0]:
%sql
select *
from `data-pipeline`.default.event_log_status_order

In [0]:
%sql
-- extract info from the event log detail column (a json string column)
select 
  id,
  event_type,
  details,
  details:flow_progress, -- you can query json string column using ':'
  details:user_action
from `data-pipeline`.default.event_log_status_order
    

#### You can build a pipeline monitoring dashboard using the event log table

Below query creates the # of records passing/failing the expectations/constraints from event log table. This pattern is commonly used to build pipeline monitoring dashboards that track data quality over time, alert on high failure rates, or identify which datasets need attention.

In [0]:
%sql
    
-- below is an example of using the event log to show the number of records passing/failing the constraints/expectations
CREATE OR REPLACE TEMPORARY VIEW dq_source_vw AS
SELECT explode(
  from_json(details:flow_progress:data_quality:expectations,
    "array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>"
  ) -- from_json() converts the JSON string into a structured array with this schema
) as row_expectations -- explode() transforms one row with an array of expectations into multiple rows (one per expectation)
FROM `data-pipeline`.default.event_log_status_order
WHERE event_type = 'flow_progress'; -- these events are emitted when data flows through a dataset and expectations are evaluated

-- view the data
select
  row_expectations.dataset as dataset,
  row_expectations.name as expectation,
  sum(row_expectations.passed_records) as passingrecords,
  sum(row_expectations.failed_records) as failed_records
from dq_source_vw
group by row_expectations.dataset, row_expectations.name 
-- combines results across all pipeline runs
order by dataset
   

## CDC Demo

In [0]:
copy_files(copy_from_path = "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/stream_json", 
           copy_to_path = "/Volumes/workspace/default/test_volume/test_directory/customers/", 
           n = 1)

Notice that our source table has a column called 'operation' to track the data change [update, new, delete]

In [0]:
%sql
select * 
from read_files('/Volumes/workspace/default/test_volume/test_directory/customers/',
format => 'json')
order by operation

In [0]:
%sql
-- check the customer_silver table created by the pipeline
select * 
from `data-pipeline`.silver_db.customers_silver
where customer_id in (23225, 23617) -- the rows that will be updated in the next batch update, where 23225 is update row, 23617 will be a delete row

In [0]:
# add data to the customer volume
copy_files(copy_from_path = "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/stream_json", 
           copy_to_path = "/Volumes/workspace/default/test_volume/test_directory/customers/", 
           n = 2)

In [0]:
%sql
select *
from read_files('/Volumes/workspace/default/test_volume/test_directory/customers/01.json',
format => 'json')

In [0]:
%sql
-- check the customer_silver table created again after running the pipeline to see the updates 
select *
from `data-pipeline`.silver_db.customers_silver
where customer_id in (23225, 23617) -- 23225 is updated, and 23617 is gone